In [1]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain.prompts import PromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain_ollama.llms import OllamaLLM

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
embedding_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
vector_store_name = "srh_index_store_faiss"
llm_name = 'llama3'

In [3]:
with open('srh_websites.txt') as f:
    sites_to_scrape = f.read().splitlines()

In [4]:
reader = WebBaseLoader(sites_to_scrape)
documents = reader.load()

In [5]:
cleaned_data = [doc.page_content.strip().replace("\n", " ") for doc in documents]

In [6]:
# Load Hugging Face embedding model
hf_embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

In [7]:
vector_store = FAISS.from_documents(documents, hf_embeddings)

In [8]:
vector_store.save_local(vector_store_name)

In [9]:
vector_store = FAISS.load_local(vector_store_name, hf_embeddings, allow_dangerous_deserialization=True)

In [10]:
raw_prompt = PromptTemplate.from_template(
    """ 
    <s>[INST] You are interested in studying at SRH Hochschule, Heidelberg and you need more information about the courses offered. If you do not have an answer from the provided information 
              say so. [/INST] </s>
    [INST] {input}
           Context: {context}
           Answer:
    [/INST]
"""
)

In [11]:
llm = OllamaLLM(model=llm_name, temperature=0)

In [12]:
retriever = vector_store.as_retriever()

document_chain = create_stuff_documents_chain(llm, raw_prompt)

chain_csv = create_retrieval_chain(retriever, document_chain)

In [13]:
def generate_response(message, history):
    response = chain_csv.invoke({"input": message})
    return response['answer']

In [14]:
import gradio as gr

gr.ChatInterface(generate_response, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
